# 01 - Download LibriSpeech

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/01_download_librispeech.ipynb)

Pulls the LibriSpeech audio subsets into the Colab VM's **local scratch** (`/content/data`, fast and ~100 GB, but wiped when the session ends). Downloads are resumable across dropped sessions, md5-verified against OpenSLR's official checksums, and extracted atomically, so a session that dies mid-extract cannot leave a half-populated subset behind. A **CPU runtime is enough**; do not spend GPU units on this.

This notebook is a thin launcher: the real code lives in `training/src/training/data/librispeech.py` in the repo. It is stdlib-only, so it runs on a stock Colab runtime without installing the ML stack.

What you get at the end:

- `/content/data/LibriSpeech/<subset>/<speaker>/<chapter>/` with the `.flac` files and one `*.trans.txt` per chapter
- corpus metadata (`SPEAKERS.TXT`, `CHAPTERS.TXT`, `BOOKS.TXT`, ...) in `/content/data/LibriSpeech/`
- a per-subset sanity table checked against the official utterance counts

Each subset prints its download-plus-extract time. Note it in the logbook; the session plan's time estimates are guesses until someone measures them.

## 1. Pull the repo into the VM

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
TRAINING_DIR = os.path.join(REPO_DIR, "training")

# Public repo, so no auth needed. Re-running the cell just fast-forwards.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Editable install of the training package (src layout: training/src/training/).
# Also installs whatever dependencies pyproject.toml lists as stages land.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", TRAINING_DIR], check=True)

# An editable install registers the package through a .pth file, and Python only
# reads those at interpreter start. This kernel is already running, so put src/
# on sys.path directly; without this, `import training` fails until a runtime
# restart.
sys.path.insert(0, os.path.join(TRAINING_DIR, "src"))
importlib.invalidate_caches()

pkg = importlib.import_module("training")
print("repo ready:", REPO_DIR)
print("package   :", pkg.__file__)

## 2. Download the audio into `/content/data`

`train-clean-100` is the acoustic-model training set to start with; `dev-clean` and `test-clean` are for decode-weight tuning and final eval. The small sets go first so a dropped session still leaves something usable and the pipeline proves itself in a minute before the 6.3 GB pull. To scale up later (if the WER target needs it), add `train-clean-360` and/or `train-other-500` to the list. The language-model text corpus is a separate pull (see the last cell); it is not needed to train the acoustic model.

Re-running this cell is safe: already-extracted subsets are skipped, and a partial `.tar.gz` from a dropped session is resumed rather than restarted.

In [ ]:
from training.data.librispeech import download

SUBSETS = ["dev-clean", "test-clean", "train-clean-100"]
DATA_DIR = "/content/data"  # local scratch, NOT Drive (Drive random reads starve the GPU)

subset_dirs = download(SUBSETS, dest=DATA_DIR)
subset_dirs

## 3. Verify what landed

For each subset: speakers, chapters, utterances vs. the official count (2,703 / 2,620 / 28,539 for dev-clean / test-clean / train-clean-100), every transcript line has its `.flac`, no empty transcripts, and every FLAC header says 16 kHz mono (the feature spec assumes it), with the summed duration next to the official hours. The last two lines list any transcript characters outside A-Z, space and apostrophe (input to the 31-symbol vocab decision in `shared/specs/`) and the audio formats seen. Header parsing opens every file, so allow a minute or two for train-clean-100.

In [ ]:
from training.data.librispeech import format_summary, summarize

rows = [summarize(d) for d in subset_dirs]
print(format_summary(rows))
assert all(r["ok"] for r in rows), "a subset is incomplete, re-run the download cell"

In [ ]:
%%bash
echo "== disk used per subset =="
du -sh /content/data/LibriSpeech/* 2>/dev/null
echo
echo "== scratch disk headroom (train-clean-360 needs ~23 GB extracted + 23 GB tar) =="
df -h /content | tail -1

## 4. (Later) Language-model text corpus

Needed for the char-LM and KenLM word-LM stages, not for acoustic-model training. Uncomment when you get there. Files on OpenSLR 11 (checked 2026-09-14): `librispeech-lm-norm.txt.gz` (1.5 GB, the char-LM corpus and the input for a custom KenLM build), `3-gram.arpa.gz` (759 MB, unpruned), `3-gram.pruned.1e-7.arpa.gz` (34 MB), `3-gram.pruned.3e-7.arpa.gz` (13 MB), `librispeech-vocab.txt` (200K words). The 13 MB pruned ARPA is enough for a first end-to-end decode; the full normalized text is only needed to build a custom LM.

In [ ]:
# from training.data.librispeech import download_lm
#
# download_lm(["3-gram.pruned.3e-7.arpa.gz", "librispeech-vocab.txt"], dest="/content/data/lm")